# 🎯 Ćwiczenie 4: Przygotowanie danych i budowanie modeli ML z scikit-learn

**Kurs:** Python w Zastosowaniach Big Data — PJATK  
**Data:** 26.04.2026  
**Czas realizacji:** ⏱️ 90 minut  
**Poziom trudności:** ⭐⭐ (Intermediate)  

---

## 📚 Cele kształcenia

Po ukończeniu tego notebooka będziesz potrafił/a:

1. **Eksplorować i diagnozować** problemy jakościowe w surowych danych medycznych
2. **Czyścić dane** — obsługiwać brakujące wartości, niespójne kategorie i wartości odstające (outliers)
3. **Tworzyć nowe cechy** (feature engineering) i kodować zmienne kategoryczne
4. **Budować Pipeline** przetwarzania danych z `ColumnTransformer` i `Pipeline` z scikit-learn
5. **Trenować, porównywać i ewaluować** modele klasyfikacji (LogisticRegression, DecisionTree, RandomForest)

## 📋 Wymagania wstępne

- Podstawy Pythona (funkcje, pętle, warunki)
- Podstawowa znajomość pandas i NumPy
- Podstawy statystyki opisowej (średnia, mediana, odchylenie standardowe)

---

> 💡 **Wskazówka:** Uruchamiaj komórki po kolei od góry do dołu za pomocą `Shift+Enter`.  
> Jeśli utkniesz — kliknij komórkę z podpowiedzią (💡) lub zapytaj prowadzącego!

In [ ]:
# @title 🔧 Konfiguracja środowiska — uruchom tę komórkę jako pierwszą!

!pip install -q scikit-learn pandas seaborn matplotlib numpy

# Standardowe importy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# scikit-learn — przetwarzanie danych
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# scikit-learn — modele
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# scikit-learn — metryki
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    ConfusionMatrixDisplay, RocCurveDisplay
)

# Ustawienia wyświetlania
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid', palette='muted')
warnings.filterwarnings('ignore')

# Ziarno losowości — powtarzalność wyników
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Środowisko skonfigurowane pomyślnie! Zaczynamy 🚀')

## 📖 Wprowadzenie — Przygotowanie danych w Machine Learning

W świecie Machine Learning istnieje popularne powiedzenie:

> **"Garbage in, garbage out"** 🗑️ — Jeśli wrzucisz śmieci do modelu, model wyprodukuje śmieci.

Szacuje się, że **60–80% czasu** data scientista to właśnie przygotowanie danych. Brzmi nudno? Może trochę — ale to dosłownie fundament, na którym stoi cały model. Bez czystych danych nawet najlepszy algorytm będzie bezużyteczny. Pomyśl o tym jak o gotowaniu 🍳 — nawet szef kuchni z gwiazdką Michelin nie zrobi dobrego dania ze zepsutych składników!

### 🔑 Kluczowe pojęcia

| Pojęcie | Opis | Przykład w kodzie |
|---------|------|-------------------|
| **Missing Values** | Brakujące wartości w danych (NaN, None) | `df.isnull().sum()` |
| **Outliers** | Wartości odstające — niepasujące do reszty danych | IQR: `Q3 + 1.5 * IQR` |
| **Feature Engineering** | Tworzenie nowych cech z istniejących kolumn | `df['bmi_cat'] = pd.cut(df['bmi'], ...)` |
| **Encoding** | Zamiana zmiennych kategorycznych na liczby | `OneHotEncoder`, `LabelEncoder` |
| **Scaling** | Normalizacja zakresów wartości liczbowych | `StandardScaler`, `MinMaxScaler` |
| **Pipeline** | Łańcuch kroków przetwarzania w jednym obiekcie | `Pipeline([('scaler', ...), ('model', ...)])` |
| **Train/Test Split** | Podział danych na zbiór treningowy i testowy | `train_test_split(X, y, test_size=0.2)` |
| **Cross-Validation** | Wielokrotna walidacja modelu na różnych podzbiorach | `cross_val_score(model, X, y, cv=5)` |

### 🏥 Kontekst biznesowy

Pracujesz jako junior data scientist w **HealthTech Polska** — startupie, który buduje system przewidywania ryzyka chorób serca ❤️. Twój zespół otrzymał surowe dane medyczne od szpitali partnerskich.

**Problem:** dane są brudne, niekompletne i wymagają solidnego przygotowania przed budowaniem modeli ML.

Twoje zadanie na dziś: przejść pełny proces **dirty data → clean data → model → ewaluacja**.

Gotowy/a? Zaczynamy! 💪

## 📊 Generowanie danych — uruchom bez modyfikacji

Poniżej wygenerujemy syntetyczny zbiór danych medycznych z **celowo wprowadzonymi problemami jakościowymi** — dokładnie tak, jak wyglądają dane w prawdziwym życiu. Uruchom poniższą komórkę bez zmian.

In [ ]:
# @title 📊 Generowanie brudnych danych medycznych — uruchom bez modyfikacji

np.random.seed(RANDOM_STATE)
N = 5000

# --- Generowanie bazowych, poprawnych danych ---
patient_ids = np.arange(1, N + 1)

# Wiek: normalny rozkład, średnia 55, odchylenie 15
age = np.random.normal(55, 15, N).round(1)
# Celowo wstawiamy kilka wartości ujemnych (błędy w danych)
error_idx_age = np.random.choice(N, size=30, replace=False)
age[error_idx_age] = np.random.uniform(-10, -1, 30).round(1)
# Celowo wstawiamy braki
null_idx_age = np.random.choice(N, size=200, replace=False)
age = age.astype(object)
age[null_idx_age] = np.nan

# Płeć: M/F z celowo niespójnymi wariantami
gender_clean = np.random.choice(['M', 'F'], N, p=[0.55, 0.45])
gender = gender_clean.astype(object).copy()
# Podmieniamy losowe wpisy na pełne nazwy
mix_idx_gender = np.random.choice(N, size=150, replace=False)
for idx in mix_idx_gender:
    if gender[idx] == 'M':
        gender[idx] = 'Mężczyzna'
    else:
        gender[idx] = 'Kobieta'
# Celowo wstawiamy braki
null_idx_gender = np.random.choice(N, size=100, replace=False)
gender[null_idx_gender] = np.nan

# Ciśnienie skurczowe: normalne ~130, odchylenie 20
bp_sys = np.random.normal(130, 20, N).round(1)
# Celowe outliery — absurdalne wartości
outlier_idx_bp = np.random.choice(N, size=20, replace=False)
bp_sys[outlier_idx_bp] = np.random.choice([500, 450, 600, 0, -10], 20)
null_idx_bp_sys = np.random.choice(N, size=180, replace=False)
bp_sys = bp_sys.astype(object)
bp_sys[null_idx_bp_sys] = np.nan

# Ciśnienie rozkurczowe: normalne ~80, odchylenie 12
bp_dia = np.random.normal(80, 12, N).round(1)
null_idx_bp_dia = np.random.choice(N, size=150, replace=False)
bp_dia = bp_dia.astype(object)
bp_dia[null_idx_bp_dia] = np.nan

# Cholesterol: normalne ~220, odchylenie 40
cholesterol = np.random.normal(220, 40, N).round(1)
null_idx_chol = np.random.choice(N, size=250, replace=False)
cholesterol = cholesterol.astype(object)
cholesterol[null_idx_chol] = np.nan

# Cukier we krwi: normalne ~110, odchylenie 30 — z celowo wmieszanym tekstem
blood_sugar_vals = np.random.normal(110, 30, N).round(1)
blood_sugar = blood_sugar_vals.astype(object).copy()
# Losowo podmieniamy wartości liczbowe na tekstowe
text_idx_bs = np.random.choice(N, size=80, replace=False)
for idx in text_idx_bs:
    blood_sugar[idx] = np.random.choice(['high', 'low', 'normal'])
null_idx_bs = np.random.choice(N, size=160, replace=False)
blood_sugar[null_idx_bs] = np.nan

# BMI: normalne ~27, odchylenie 6
bmi = np.random.normal(27, 6, N).round(1)
# Celowe wartości ekstremalne
extreme_idx_bmi = np.random.choice(N, size=15, replace=False)
bmi[extreme_idx_bmi] = np.random.choice([0, -5, 80, 95, 100], 15)
null_idx_bmi = np.random.choice(N, size=130, replace=False)
bmi = bmi.astype(object)
bmi[null_idx_bmi] = np.nan

# Palenie: Tak/Nie z wmieszanymi 1/0
smoking_clean = np.random.choice(['Tak', 'Nie'], N, p=[0.3, 0.7])
smoking = smoking_clean.astype(object).copy()
mix_idx_smoking = np.random.choice(N, size=120, replace=False)
for idx in mix_idx_smoking:
    smoking[idx] = '1' if smoking[idx] == 'Tak' else '0'
null_idx_smoking = np.random.choice(N, size=90, replace=False)
smoking[null_idx_smoking] = np.nan

# Godziny ćwiczeń tygodniowo: gamma rozkład
exercise = np.random.gamma(2, 1.5, N).round(1)
null_idx_ex = np.random.choice(N, size=170, replace=False)
exercise = exercise.astype(object)
exercise[null_idx_ex] = np.nan

# Zmienna docelowa (target): choroba serca
# Prawdopodobieństwo zależy od czynników ryzyka
base_prob = np.full(N, 0.15)
# Wyższy wiek -> wyższe ryzyko
age_numeric = pd.to_numeric(age, errors='coerce')
base_prob = np.where(age_numeric > 60, base_prob + 0.15, base_prob)
base_prob = np.where(age_numeric > 70, base_prob + 0.10, base_prob)
# Palenie -> wyższe ryzyko
base_prob = np.where((smoking == 'Tak') | (smoking == '1'), base_prob + 0.12, base_prob)
# Wysokie ciśnienie -> wyższe ryzyko
bp_numeric = pd.to_numeric(bp_sys, errors='coerce')
base_prob = np.where(bp_numeric > 150, base_prob + 0.10, base_prob)
# Ograniczenie do [0, 1]
base_prob = np.clip(base_prob, 0.05, 0.85)
heart_disease = np.array([np.random.binomial(1, p) for p in base_prob])

# --- Składanie DataFrame ---
df_raw = pd.DataFrame({
    'patient_id': patient_ids,
    'age': age,
    'gender': gender,
    'blood_pressure_systolic': bp_sys,
    'blood_pressure_diastolic': bp_dia,
    'cholesterol': cholesterol,
    'blood_sugar': blood_sugar,
    'bmi': bmi,
    'smoking': smoking,
    'exercise_hours_weekly': exercise,
    'heart_disease': heart_disease
})

print(f'📊 Wygenerowano zbiór danych: {df_raw.shape[0]} wierszy, {df_raw.shape[1]} kolumn')
print(f'🎯 Zmienna docelowa (heart_disease): {heart_disease.sum()} przypadków choroby '
      f'({heart_disease.mean()*100:.1f}%)')
print(f'\n🔍 Pierwsze 5 wierszy:')
df_raw.head()

---

## 🎓 Przykład prowadzony — Od brudnych danych do modelu

Zanim przejdziesz do samodzielnych ćwiczeń, prześledźmy razem **uproszczoną wersję** całego procesu na małym podzbiorze danych. To jak "trailer" tego, co będziesz robić w kolejnych zadaniach 🎬

In [ ]:
# @title 🎓 Przykład prowadzony — pełny mini-pipeline (uruchom i przeanalizuj)

# Krok 1: Weźmy mały podzbiór danych (3 kolumny) do demonstracji
demo = df_raw[['age', 'cholesterol', 'heart_disease']].copy()

# Krok 2: Konwersja typów
demo['age'] = pd.to_numeric(demo['age'], errors='coerce')
demo['cholesterol'] = pd.to_numeric(demo['cholesterol'], errors='coerce')

print('=== PRZED czyszczeniem ===')
print(f'Braki w danych:\n{demo.isnull().sum()}')
print(f'Ujemne wartości age: {(demo["age"] < 0).sum()}')

# Krok 3: Usuwamy ujemne wartości wieku (zamieniamy na NaN)
demo.loc[demo['age'] < 0, 'age'] = np.nan

# Krok 4: Imputacja braków medianą
imputer = SimpleImputer(strategy='median')
demo[['age', 'cholesterol']] = imputer.fit_transform(demo[['age', 'cholesterol']])

print('\n=== PO czyszczeniu ===')
print(f'Braki w danych:\n{demo.isnull().sum()}')

# Krok 5: Podział na cechy (X) i zmienną docelową (y)
X_demo = demo[['age', 'cholesterol']]
y_demo = demo['heart_disease']

# Krok 6: Train/test split
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=RANDOM_STATE, stratify=y_demo
)
print(f'\nZbiór treningowy: {X_train_d.shape[0]} próbek')
print(f'Zbiór testowy:    {X_test_d.shape[0]} próbek')

# Krok 7: Skalowanie + model w Pipeline
demo_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

demo_pipeline.fit(X_train_d, y_train_d)
y_pred_d = demo_pipeline.predict(X_test_d)

# Krok 8: Ewaluacja
print(f'\n=== WYNIKI MODELU ===')
print(f'Accuracy:  {accuracy_score(y_test_d, y_pred_d):.3f}')
print(f'Precision: {precision_score(y_test_d, y_pred_d, zero_division=0):.3f}')
print(f'Recall:    {recall_score(y_test_d, y_pred_d, zero_division=0):.3f}')
print(f'F1-score:  {f1_score(y_test_d, y_pred_d, zero_division=0):.3f}')

print('\n✅ Mini-pipeline zakończony! Teraz Twoja kolej — w pełnej wersji 💪')

---

## ✏️ Ćwiczenie 1: Eksploracja brudnych danych

**Trudność:** ⭐

Zanim cokolwiek wyczyścisz, musisz **zrozumieć, z czym masz do czynienia**. To jak wizyta u lekarza — najpierw diagnoza, potem leczenie 🩺

**Zadanie:**
1. Wyświetl kształt (shape) zbioru danych `df_raw`
2. Wyświetl typy danych (`dtypes`)
3. Policz brakujące wartości w każdej kolumnie
4. Wyświetl statystyki opisowe (`describe()`)
5. Zapisz liczbę brakujących wartości w zmiennej `missing_counts` (jako Series)

**Oczekiwany wynik:** Series z liczbą brakujących wartości per kolumna.

In [ ]:
# ✏️ Ćwiczenie 1: Eksploracja brudnych danych

# Pracujemy na kopii surowych danych
df = df_raw.copy()

# TODO: Wyświetl kształt zbioru danych
print('Kształt zbioru danych:', ...)

# TODO: Wyświetl typy danych
print('\nTypy danych:')
print(...)

# TODO: Policz i wyświetl brakujące wartości (braki = NaN)
missing_counts = ...  # Zastąp ... kodem
print('\nBrakujące wartości:')
print(missing_counts)

# TODO: Wyświetl statystyki opisowe
print('\nStatystyki opisowe:')
...

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 1
print("""
💡 Podpowiedzi:
1. Kształt: df.shape
2. Typy danych: df.dtypes
3. Brakujące wartości: df.isnull().sum()
4. Statystyki opisowe: df.describe() — dodaj include='all' żeby zobaczyć też kolumny kategoryczne

Uwaga: wiele kolumn ma typ 'object' — to oznacza, że zawierają mieszankę typów!
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 1
try:
    assert missing_counts is not ..., '❌ Nie uzupełniłeś/aś zmiennej missing_counts'
    assert isinstance(missing_counts, pd.Series), '❌ missing_counts powinno być typu pd.Series'
    assert len(missing_counts) == df.shape[1], f'❌ Oczekiwano {df.shape[1]} kolumn, dostałem {len(missing_counts)}'
    assert missing_counts.sum() > 0, '❌ Powinny być jakieś brakujące wartości — coś poszło nie tak'
    print(f'✅ Ćwiczenie 1 zaliczone! Znaleziono {missing_counts.sum()} brakujących wartości łącznie 🎉')
    print(f'📊 Kolumna z największą liczbą braków: {missing_counts.idxmax()} ({missing_counts.max()} braków)')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 2: Obsługa brakujących wartości

**Trudność:** ⭐⭐

Brakujące wartości (missing values) to chleb powszedni data scientista. Mamy kilka strategii:
- **Usunięcie** wierszy/kolumn — proste, ale tracimy dane 😢
- **Imputacja medianą** — dla zmiennych numerycznych (odporna na outliers)
- **Imputacja modą** (most_frequent) — dla zmiennych kategorycznych
- **Imputacja średnią** — dla zmiennych numerycznych (wrażliwa na outliers)

**Zadanie:**
1. Skonwertuj kolumny numeryczne (`age`, `blood_pressure_systolic`, `blood_pressure_diastolic`, `cholesterol`, `bmi`, `exercise_hours_weekly`) na typ float za pomocą `pd.to_numeric(..., errors='coerce')`
2. Użyj `SimpleImputer(strategy='median')` do uzupełnienia braków w kolumnach **numerycznych**
3. Użyj `SimpleImputer(strategy='most_frequent')` do uzupełnienia braków w kolumnach **kategorycznych** (`gender`, `smoking`)
4. Sprawdź, czy po imputacji nie ma już braków

**Oczekiwany wynik:** DataFrame `df` bez żadnych wartości NaN w powyższych kolumnach.

In [ ]:
# ✏️ Ćwiczenie 2: Obsługa brakujących wartości

# Lista kolumn numerycznych do konwersji
numeric_cols = ['age', 'blood_pressure_systolic', 'blood_pressure_diastolic',
                'cholesterol', 'bmi', 'exercise_hours_weekly']
categorical_cols = ['gender', 'smoking']

# TODO: Krok 1 — Konwertuj kolumny numeryczne na float
# Wskazówka: użyj pętli for i pd.to_numeric(..., errors='coerce')
for col in numeric_cols:
    df[col] = ...  # Zastąp ...

# TODO: Krok 2 — Imputacja braków w kolumnach numerycznych (mediana)
imputer_num = SimpleImputer(strategy=...)  # Zastąp ...
df[numeric_cols] = imputer_num.fit_transform(df[numeric_cols])

# TODO: Krok 3 — Imputacja braków w kolumnach kategorycznych (moda)
imputer_cat = SimpleImputer(strategy=...)  # Zastąp ...
df[categorical_cols] = imputer_cat.fit_transform(df[categorical_cols])

# Sprawdzenie wyniku
remaining_nulls = df[numeric_cols + categorical_cols].isnull().sum().sum()
print(f'Pozostałe braki w kolumnach po imputacji: {remaining_nulls}')
df[numeric_cols].describe()

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 2
print("""
💡 Podpowiedzi:
1. Konwersja: df[col] = pd.to_numeric(df[col], errors='coerce')
   errors='coerce' zamieni niepoprawne wartości (np. tekst) na NaN

2. Imputacja numeryczna:
   imputer_num = SimpleImputer(strategy='median')
   df[numeric_cols] = imputer_num.fit_transform(df[numeric_cols])

3. Imputacja kategoryczna:
   imputer_cat = SimpleImputer(strategy='most_frequent')
   df[categorical_cols] = imputer_cat.fit_transform(df[categorical_cols])
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 2
try:
    target_cols = numeric_cols + categorical_cols
    nulls_after = df[target_cols].isnull().sum().sum()
    assert nulls_after == 0, f'❌ Wciąż masz {nulls_after} brakujących wartości w przetworzonych kolumnach'
    assert df['age'].dtype in [np.float64, np.float32, float], '❌ Kolumna age powinna być typu float'
    assert df['cholesterol'].dtype in [np.float64, np.float32, float], '❌ Kolumna cholesterol powinna być typu float'
    print(f'✅ Ćwiczenie 2 zaliczone! Wszystkie braki uzupełnione 🎉')
    print(f'📊 Mediana wieku po imputacji: {df["age"].median():.1f} lat')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 3: Czyszczenie i naprawianie danych

**Trudność:** ⭐⭐

Teraz czas na "chirurgię" 🔪 — musimy naprawić niespójne dane:

| Problem | Przykład | Rozwiązanie |
|---------|----------|-------------|
| Niespójna płeć | `'M'`, `'Mężczyzna'` | Ujednolicić do `'M'`/`'F'` |
| Niespójne palenie | `'Tak'`, `'1'` | Ujednolicić do `'Tak'`/`'Nie'` |
| Ujemny wiek | `-5.3` | Zamienić na medianę lub usunąć |
| Absurdalne ciśnienie | `500`, `0` | Usunąć/zamienić na NaN, potem imputować |
| Ekstremalny BMI | `100`, `-5` | Filtrować zakresem (14–60) |

**Zadanie:**
1. Ujednolić kolumnę `gender`: `'Mężczyzna'` → `'M'`, `'Kobieta'` → `'F'`
2. Ujednolić kolumnę `smoking`: `'1'` → `'Tak'`, `'0'` → `'Nie'`
3. Zamienić ujemne wartości `age` na medianę kolumny
4. Zamienić nierealistyczne wartości `blood_pressure_systolic` (< 60 lub > 250) na medianę
5. Zamienić nierealistyczne wartości `bmi` (< 10 lub > 60) na medianę
6. Zapisz liczbę naprawionych wierszy łącznie we wszystkich kolumnach w zmiennej `total_fixed`

**Oczekiwany wynik:** Zmienne `total_fixed` z liczbą naprawionych wierszy.

In [ ]:
# ✏️ Ćwiczenie 3: Czyszczenie i naprawianie danych

fixed_count = 0  # Licznik naprawionych wartości

# TODO: Krok 1 — Ujednolicenie kolumny gender
# Zamień 'Mężczyzna' na 'M' i 'Kobieta' na 'F'
count_before = (df['gender'].isin(['Mężczyzna', 'Kobieta'])).sum()
df['gender'] = df['gender'].replace(...)  # Zastąp ... słownikiem mapowania
fixed_count += count_before

# TODO: Krok 2 — Ujednolicenie kolumny smoking
# Zamień '1' na 'Tak' i '0' na 'Nie'
count_before = (df['smoking'].isin(['1', '0'])).sum()
df['smoking'] = df['smoking'].replace(...)  # Zastąp ... słownikiem mapowania
fixed_count += count_before

# TODO: Krok 3 — Napraw ujemne wartości age
# Zamień wartości age < 0 na medianę kolumny (obliczoną z poprawnych wartości)
mask_neg_age = ...  # Zastąp ... warunkiem
fixed_count += mask_neg_age.sum()
median_age = df.loc[df['age'] > 0, 'age'].median()
df.loc[mask_neg_age, 'age'] = median_age

# TODO: Krok 4 — Napraw nierealistyczne ciśnienie skurczowe
# Zamień wartości < 60 lub > 250 na medianę
mask_bad_bp = ...  # Zastąp ... warunkiem
fixed_count += mask_bad_bp.sum()
median_bp = df.loc[~mask_bad_bp, 'blood_pressure_systolic'].median()
df.loc[mask_bad_bp, 'blood_pressure_systolic'] = median_bp

# TODO: Krok 5 — Napraw nierealistyczne BMI
# Zamień wartości < 10 lub > 60 na medianę
mask_bad_bmi = ...  # Zastąp ... warunkiem
fixed_count += mask_bad_bmi.sum()
median_bmi = df.loc[~mask_bad_bmi, 'bmi'].median()
df.loc[mask_bad_bmi, 'bmi'] = median_bmi

total_fixed = fixed_count
print(f'🔧 Łącznie naprawiono {total_fixed} wartości')
print(f'\nUnikalne wartości gender: {df["gender"].unique()}')
print(f'Unikalne wartości smoking: {df["smoking"].unique()}')
print(f'Zakres age: [{df["age"].min():.1f}, {df["age"].max():.1f}]')
print(f'Zakres BP systolic: [{df["blood_pressure_systolic"].min():.1f}, {df["blood_pressure_systolic"].max():.1f}]')
print(f'Zakres BMI: [{df["bmi"].min():.1f}, {df["bmi"].max():.1f}]')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 3
print("""
💡 Podpowiedzi:

1. Mapowanie wartości:
   df['gender'] = df['gender'].replace({'Mężczyzna': 'M', 'Kobieta': 'F'})
   df['smoking'] = df['smoking'].replace({'1': 'Tak', '0': 'Nie'})

2. Maski dla niepoprawnych wartości:
   mask_neg_age = df['age'] < 0
   mask_bad_bp = (df['blood_pressure_systolic'] < 60) | (df['blood_pressure_systolic'] > 250)
   mask_bad_bmi = (df['bmi'] < 10) | (df['bmi'] > 60)

3. Zamiana na medianę:
   median_val = df.loc[~mask, 'kolumna'].median()
   df.loc[mask, 'kolumna'] = median_val
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 3
try:
    # Sprawdzenie gender
    unique_gender = set(df['gender'].unique())
    assert unique_gender <= {'M', 'F'}, f'❌ Gender powinien mieć tylko M/F, masz: {unique_gender}'

    # Sprawdzenie smoking
    unique_smoking = set(df['smoking'].unique())
    assert unique_smoking <= {'Tak', 'Nie'}, f'❌ Smoking powinien mieć Tak/Nie, masz: {unique_smoking}'

    # Sprawdzenie age
    assert df['age'].min() >= 0, f'❌ Wciąż są ujemne wartości age: min = {df["age"].min()}'

    # Sprawdzenie BP
    assert df['blood_pressure_systolic'].min() >= 60, '❌ Wciąż są zbyt niskie wartości ciśnienia'
    assert df['blood_pressure_systolic'].max() <= 250, '❌ Wciąż są zbyt wysokie wartości ciśnienia'

    # Sprawdzenie BMI
    assert df['bmi'].min() >= 10, '❌ Wciąż są zbyt niskie wartości BMI'
    assert df['bmi'].max() <= 60, '❌ Wciąż są zbyt wysokie wartości BMI'

    assert total_fixed > 0, '❌ total_fixed powinno być > 0'

    print(f'✅ Ćwiczenie 3 zaliczone! Naprawiono {total_fixed} wartości 🔧🎉')
    print(f'📊 Dane po czyszczeniu wyglądają dużo lepiej!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 4: Feature Engineering

**Trudność:** ⭐⭐

Feature engineering to **sztuka tworzenia nowych cech** z istniejących danych. To jak gotowanie — masz składniki (surowe kolumny), a z nich tworzysz danie (nowe, bardziej informacyjne cechy) 🍕

Dobre cechy mogą **znacząco poprawić** jakość modelu!

**Zadanie:**
1. Utwórz kolumnę `bmi_category` na podstawie BMI:
   - BMI < 18.5 → `'niedowaga'`
   - 18.5 ≤ BMI < 25 → `'norma'`
   - 25 ≤ BMI < 30 → `'nadwaga'`
   - BMI ≥ 30 → `'otyłość'`
2. Utwórz kolumnę `age_group`:
   - age < 40 → `'młody'`
   - 40 ≤ age < 60 → `'średni'`
   - age ≥ 60 → `'senior'`
3. Utwórz kolumnę `bp_ratio` = `blood_pressure_systolic` / `blood_pressure_diastolic`
4. Zakoduj zmienne kategoryczne (`gender`, `smoking`, `bmi_category`, `age_group`) za pomocą `pd.get_dummies()` — zapisz wynik w `df_encoded`

**Oczekiwany wynik:** DataFrame `df_encoded` z nowymi cechami i kolumnami one-hot.

In [ ]:
# ✏️ Ćwiczenie 4: Feature Engineering

# TODO: Krok 1 — Utwórz kolumnę bmi_category
# Wskazówka: użyj pd.cut() z bins i labels
df['bmi_category'] = pd.cut(
    df['bmi'],
    bins=...,  # Zastąp ... listą granic przedziałów
    labels=...,  # Zastąp ... listą nazw kategorii
    right=False
)

# TODO: Krok 2 — Utwórz kolumnę age_group
# Wskazówka: użyj pd.cut() z bins i labels
df['age_group'] = pd.cut(
    df['age'],
    bins=...,  # Zastąp ...
    labels=...,  # Zastąp ...
    right=False
)

# TODO: Krok 3 — Utwórz kolumnę bp_ratio
df['bp_ratio'] = ...  # Zastąp ...

# TODO: Krok 4 — One-hot encoding za pomocą pd.get_dummies()
cols_to_encode = ['gender', 'smoking', 'bmi_category', 'age_group']
df_encoded = pd.get_dummies(df, columns=..., drop_first=...)  # Zastąp ...

print(f'📊 Kształt df po feature engineering: {df.shape}')
print(f'📊 Kształt df_encoded (po one-hot): {df_encoded.shape}')
print(f'\nNowe kolumny: {[c for c in df_encoded.columns if c not in df_raw.columns]}')
df_encoded.head()

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 4
print("""
💡 Podpowiedzi:

1. BMI category z pd.cut():
   bins = [0, 18.5, 25, 30, 100]
   labels = ['niedowaga', 'norma', 'nadwaga', 'otyłość']
   df['bmi_category'] = pd.cut(df['bmi'], bins=bins, labels=labels, right=False)

2. Age group:
   bins = [0, 40, 60, 120]
   labels = ['młody', 'średni', 'senior']

3. BP ratio — po prostu dzielenie:
   df['bp_ratio'] = df['blood_pressure_systolic'] / df['blood_pressure_diastolic']

4. One-hot encoding:
   df_encoded = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)
   drop_first=True — zapobiega 'dummy variable trap'
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 4
try:
    # Sprawdzenie bmi_category
    assert 'bmi_category' in df.columns, '❌ Brak kolumny bmi_category'
    valid_bmi_cats = {'niedowaga', 'norma', 'nadwaga', 'otyłość'}
    actual_cats = set(df['bmi_category'].dropna().unique())
    assert actual_cats <= valid_bmi_cats, f'❌ Nieoczekiwane kategorie BMI: {actual_cats - valid_bmi_cats}'

    # Sprawdzenie age_group
    assert 'age_group' in df.columns, '❌ Brak kolumny age_group'
    valid_age_groups = {'młody', 'średni', 'senior'}
    actual_groups = set(df['age_group'].dropna().unique())
    assert actual_groups <= valid_age_groups, f'❌ Nieoczekiwane grupy wiekowe: {actual_groups - valid_age_groups}'

    # Sprawdzenie bp_ratio
    assert 'bp_ratio' in df.columns, '❌ Brak kolumny bp_ratio'
    assert df['bp_ratio'].mean() > 1.0, '❌ bp_ratio powinno być > 1 (skurczowe > rozkurczowe)'

    # Sprawdzenie df_encoded
    assert df_encoded is not ..., '❌ Nie utworzono df_encoded'
    assert df_encoded.shape[1] > df_raw.shape[1], '❌ df_encoded powinien mieć więcej kolumn niż df_raw'

    print(f'✅ Ćwiczenie 4 zaliczone! Utworzono {df_encoded.shape[1] - df_raw.shape[1]} nowych kolumn 🎉')
    print(f'📊 Rozkład BMI: {df["bmi_category"].value_counts().to_dict()}')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 5: Skalowanie i przygotowanie do modelu

**Trudność:** ⭐⭐

Czas na ostatni krok przygotowania danych! Wiele algorytmów ML (np. Logistic Regression) działa lepiej, gdy cechy są **znormalizowane** — tzn. mają podobny zakres wartości.

**StandardScaler** przekształca dane tak, że mają średnią = 0 i odchylenie standardowe = 1.

Użyjemy też **`ColumnTransformer`** — potężnego narzędzia z scikit-learn, które pozwala stosować **różne transformacje** do różnych kolumn w jednym kroku!

**Zadanie:**
1. Przygotuj macierz cech `X` (wszystkie kolumny oprócz `patient_id`, `heart_disease`, `blood_sugar` i oryginalnych kolumn kategorycznych) i wektor docelowy `y` (`heart_disease`)
2. Wykonaj train/test split (80/20) ze stratyfikacją i `random_state=42`
3. Zbuduj `ColumnTransformer` ze `StandardScaler` na kolumnach numerycznych
4. Dopasuj (fit) transformer na zbiorze treningowym i przetransformuj oba zbiory

**Oczekiwany wynik:** `X_train`, `X_test`, `y_train`, `y_test` gotowe do trenowania modeli.

In [ ]:
# ✏️ Ćwiczenie 5: Skalowanie i przygotowanie do modelu

# Kolumny, które usuwamy (nie są cechami modelu)
drop_cols = ['patient_id', 'heart_disease', 'blood_sugar',
             'gender', 'smoking', 'bmi_category', 'age_group']

# TODO: Krok 1 — Przygotuj X i y
X = df_encoded.drop(columns=...)  # Zastąp ... listą kolumn do usunięcia
y = df_encoded[...]  # Zastąp ... nazwą kolumny docelowej

print(f'Cechy (X): {X.shape}')
print(f'Target (y): {y.shape}, rozkład: {y.value_counts().to_dict()}')

# TODO: Krok 2 — Train/test split (80/20, stratyfikacja, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=...,  # Zastąp ...
    random_state=...,  # Zastąp ...
    stratify=...  # Zastąp ...
)

print(f'\nZbiór treningowy: {X_train.shape[0]} próbek')
print(f'Zbiór testowy:    {X_test.shape[0]} próbek')

# TODO: Krok 3 — Zidentyfikuj kolumny numeryczne do skalowania
# (te, które nie są typu bool / one-hot encoded)
num_features = X.select_dtypes(include=[np.number]).columns.tolist()

# TODO: Krok 4 — Zbuduj ColumnTransformer ze StandardScaler
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ...)  # Zastąp ... listą kolumn numerycznych
    ],
    remainder='passthrough'  # Zostawiamy pozostałe kolumny bez zmian
)

# Dopasowanie i transformacja
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

print(f'\n✅ Dane przygotowane do modelu!')
print(f'X_train_scaled shape: {X_train_scaled.shape}')
print(f'X_test_scaled shape:  {X_test_scaled.shape}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 5
print("""
💡 Podpowiedzi:

1. Przygotowanie X i y:
   # Usuń kolumny, które nie powinny wchodzić w skład cech
   cols_to_drop = [c for c in drop_cols if c in df_encoded.columns]
   X = df_encoded.drop(columns=cols_to_drop)
   y = df_encoded['heart_disease']

2. Train/test split:
   X_train, X_test, y_train, y_test = train_test_split(
       X, y, test_size=0.2, random_state=42, stratify=y
   )

3. ColumnTransformer:
   preprocessor = ColumnTransformer(
       transformers=[
           ('num', StandardScaler(), num_features)
       ],
       remainder='passthrough'
   )
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 5
try:
    assert X_train is not None, '❌ X_train nie istnieje'
    assert X_test is not None, '❌ X_test nie istnieje'
    assert y_train is not None, '❌ y_train nie istnieje'
    assert y_test is not None, '❌ y_test nie istnieje'

    # Sprawdzenie proporcji split
    total = len(X_train) + len(X_test)
    train_ratio = len(X_train) / total
    assert 0.78 < train_ratio < 0.82, f'❌ Proporcja train powinna być ~80%, jest {train_ratio:.1%}'

    # Sprawdzenie skalowania
    assert X_train_scaled is not None, '❌ X_train_scaled nie istnieje'
    assert X_train_scaled.shape[0] == X_train.shape[0], '❌ Liczba wierszy się nie zgadza'

    print(f'✅ Ćwiczenie 5 zaliczone! Dane gotowe do modelowania 🎉')
    print(f'📊 Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Cechy: {X_train.shape[1]}')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 6: Budowanie i porównanie modeli

**Trudność:** ⭐⭐⭐

Wreszcie to, na co czekaliśmy — **budowanie modeli**! 🤖

Wytrenujemy trzy popularne algorytmy klasyfikacji i porównamy ich wyniki:

| Model | Opis | Zalety |
|-------|------|--------|
| **LogisticRegression** | Klasyczny model liniowy | Szybki, interpretowalny |
| **DecisionTreeClassifier** | Drzewo decyzyjne | Łatwy do wizualizacji, uchwyci nieliniowości |
| **RandomForestClassifier** | Zespół drzew (ensemble) | Zwykle najlepszy wynik, odporny na overfitting |

**Zadanie:**
1. Wytrenuj 3 modele na `X_train_scaled` / `y_train`
2. Oblicz predykcje na `X_test_scaled`
3. Oblicz metryki: accuracy, precision, recall, F1 dla każdego modelu
4. Zbierz wyniki w DataFrame `results_df` z kolumnami: `Model`, `Accuracy`, `Precision`, `Recall`, `F1`

**Oczekiwany wynik:** Tabelka porównawcza 3 modeli.

In [ ]:
# ✏️ Ćwiczenie 6: Budowanie i porównanie modeli

# TODO: Krok 1 — Zdefiniuj modele
models = {
    'LogisticRegression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'DecisionTree': ...,  # Zastąp ... -> DecisionTreeClassifier(random_state=RANDOM_STATE)
    'RandomForest': ...   # Zastąp ... -> RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100)
}

# TODO: Krok 2 — Wytrenuj każdy model i oblicz metryki
results = []

for name, model in models.items():
    # Trenowanie modelu
    model.fit(...)  # Zastąp ... danymi treningowymi

    # Predykcja na zbiorze testowym
    y_pred = model.predict(...)  # Zastąp ... danymi testowymi

    # Obliczenie metryk
    acc = accuracy_score(y_test, y_pred)
    prec = ...  # Zastąp ... -> precision_score(y_test, y_pred, zero_division=0)
    rec = ...   # Zastąp ... -> recall_score(y_test, y_pred, zero_division=0)
    f1 = ...    # Zastąp ... -> f1_score(y_test, y_pred, zero_division=0)

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4)
    })
    print(f'✅ {name}: Accuracy={acc:.4f}, F1={f1:.4f}')

# TODO: Krok 3 — Utwórz DataFrame z wynikami
results_df = pd.DataFrame(...)  # Zastąp ...
results_df = results_df.sort_values('F1', ascending=False)

print('\n📊 Porównanie modeli:')
results_df

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 6
print("""
💡 Podpowiedzi:

1. Definicja modeli:
   'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE)
   'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100)

2. Trenowanie i predykcja:
   model.fit(X_train_scaled, y_train)
   y_pred = model.predict(X_test_scaled)

3. Metryki:
   prec = precision_score(y_test, y_pred, zero_division=0)
   rec = recall_score(y_test, y_pred, zero_division=0)
   f1 = f1_score(y_test, y_pred, zero_division=0)

4. DataFrame z wynikami:
   results_df = pd.DataFrame(results)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 6
try:
    assert results_df is not ..., '❌ Nie utworzono results_df'
    assert isinstance(results_df, pd.DataFrame), '❌ results_df powinno być DataFrame'
    assert len(results_df) == 3, f'❌ Oczekiwano 3 modeli, dostałem {len(results_df)}'

    required_cols = {'Model', 'Accuracy', 'Precision', 'Recall', 'F1'}
    assert required_cols <= set(results_df.columns), f'❌ Brakuje kolumn: {required_cols - set(results_df.columns)}'

    # Sprawdzenie sensowności wyników
    for _, row in results_df.iterrows():
        assert 0 < row['Accuracy'] <= 1, f'❌ Accuracy poza zakresem dla {row["Model"]}'
        assert 0 <= row['F1'] <= 1, f'❌ F1 poza zakresem dla {row["Model"]}'

    best = results_df.iloc[0]
    print(f'✅ Ćwiczenie 6 zaliczone! Porównano 3 modele 🎉')
    print(f'🏆 Najlepszy model wg F1: {best["Model"]} (F1={best["F1"]:.4f})')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 7: Ewaluacja i analiza błędów

**Trudność:** ⭐⭐⭐

Samo accuracy to za mało! W problemach medycznych **fałszywy negatyw** (model mówi "zdrowy", a pacjent jest chory) jest dużo gorszy niż fałszywy pozytyw. Dlatego musimy analizować wyniki głębiej 🔬

**Zadanie:**
1. Narysuj **confusion matrix** (macierz pomyłek) dla najlepszego modelu (RandomForest)
2. Narysuj **krzywą ROC** z wartością AUC
3. Wyświetl **feature importance** (ważność cech) z RandomForest — top 10 najważniejszych cech
4. Zapisz wartość AUC w zmiennej `auc_score`

**Oczekiwany wynik:** 3 wykresy (confusion matrix, ROC curve, feature importance) + wartość AUC.

In [ ]:
# ✏️ Ćwiczenie 7: Ewaluacja i analiza błędów

# Używamy modelu RandomForest z ćwiczenia 6
rf_model = models['RandomForest']
y_pred_rf = rf_model.predict(X_test_scaled)
y_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]  # Prawdopodobieństwa klasy 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# TODO: Krok 1 — Confusion Matrix
# Wskazówka: użyj ConfusionMatrixDisplay.from_predictions()
...  # Zastąp ... kodem rysującym confusion matrix na axes[0]
axes[0].set_title('Macierz pomyłek — RandomForest', fontsize=12)

# TODO: Krok 2 — Krzywa ROC
# Wskazówka: użyj RocCurveDisplay.from_predictions()
...  # Zastąp ... kodem rysującym krzywą ROC na axes[1]
axes[1].set_title('Krzywa ROC — RandomForest', fontsize=12)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)  # Linia losowego klasyfikatora

# TODO: Krok 3 — Feature Importance
# Wskazówka: rf_model.feature_importances_ zawiera ważność cech
feature_names = X.columns if hasattr(X, 'columns') else [f'feature_{i}' for i in range(X_train_scaled.shape[1])]
importances = ...  # Zastąp ... -> rf_model.feature_importances_

# Sortujemy i bierzemy top 10
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True).tail(10)
feat_imp.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 10 najważniejszych cech', fontsize=12)
axes[2].set_xlabel('Ważność (importance)')

plt.tight_layout()
plt.show()

# TODO: Krok 4 — Oblicz AUC
fpr, tpr, _ = roc_curve(y_test, y_proba_rf)
auc_score = ...  # Zastąp ... -> auc(fpr, tpr)

print(f'\n📊 AUC Score: {auc_score:.4f}')
print(f'📊 Najważniejsza cecha: {feat_imp.index[-1]} (importance={feat_imp.values[-1]:.4f})')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 7
print("""
💡 Podpowiedzi:

1. Confusion Matrix:
   ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=axes[0], cmap='Blues')

2. Krzywa ROC:
   RocCurveDisplay.from_predictions(y_test, y_proba_rf, ax=axes[1])

3. Feature Importance:
   importances = rf_model.feature_importances_

4. AUC Score:
   fpr, tpr, _ = roc_curve(y_test, y_proba_rf)
   auc_score = auc(fpr, tpr)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 7
try:
    assert auc_score is not ..., '❌ Nie obliczono auc_score'
    assert isinstance(auc_score, float), '❌ auc_score powinno być typu float'
    assert 0.5 < auc_score <= 1.0, f'❌ AUC powinno być > 0.5, masz: {auc_score:.4f}'

    assert importances is not ..., '❌ Nie przypisano importances'
    assert len(importances) == X_train_scaled.shape[1], '❌ Długość importances nie pasuje do liczby cech'

    if auc_score > 0.7:
        print(f'✅ Ćwiczenie 7 zaliczone! AUC = {auc_score:.4f} — niezły wynik! 🎉')
    else:
        print(f'✅ Ćwiczenie 7 zaliczone! AUC = {auc_score:.4f} — model działa, ale jest miejsce na poprawę 📈')

    print(f'📊 Interpretacja: model jest lepszy od losowego klasyfikatora (AUC > 0.5) ✅')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## 🏆 Bonus: Cross-Validation i Tuning hiperparametrów

**Trudność:** ⭐⭐⭐

Dla tych, co chcą więcej! 🚀

**Cross-validation** (walidacja krzyżowa) to technika, która dzieli dane treningowe na K części ("folds") i trenuje model K razy, za każdym razem używając innej części jako zbioru walidacyjnego. To daje bardziej wiarygodną ocenę jakości modelu.

**GridSearchCV** automatycznie przeszukuje przestrzeń hiperparametrów i znajduje najlepszą kombinację.

**Zadanie:**
1. Zdefiniuj siatkę hiperparametrów dla RandomForest:
   - `n_estimators`: [50, 100, 200]
   - `max_depth`: [5, 10, 20, None]
   - `min_samples_split`: [2, 5, 10]
2. Uruchom `GridSearchCV` z 5-fold cross-validation
3. Wyświetl najlepsze hiperparametry i wynik CV
4. Porównaj wynik najlepszego modelu z wynikami z ćwiczenia 6
5. Zapisz najlepszy wynik CV (mean) w zmiennej `best_cv_score`

In [ ]:
# 🏆 Bonus: Cross-Validation i Tuning hiperparametrów

# TODO: Krok 1 — Zdefiniuj siatkę hiperparametrów
param_grid = {
    'n_estimators': ...,    # Zastąp ... -> [50, 100, 200]
    'max_depth': ...,       # Zastąp ... -> [5, 10, 20, None]
    'min_samples_split': ...  # Zastąp ... -> [2, 5, 10]
}

# TODO: Krok 2 — GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=...,  # Zastąp ...
    cv=...,          # Zastąp ... -> 5
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print('🔍 Szukanie najlepszych hiperparametrów... (to może chwilę potrwać)')
grid_search.fit(X_train_scaled, y_train)

# TODO: Krok 3 — Wyświetl najlepsze hiperparametry
print(f'\n🏆 Najlepsze hiperparametry: {grid_search.best_params_}')
best_cv_score = ...  # Zastąp ... -> grid_search.best_score_
print(f'📊 Najlepszy wynik CV (F1): {best_cv_score:.4f}')

# TODO: Krok 4 — Ewaluacja na zbiorze testowym
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)

print(f'\n📊 Wyniki najlepszego modelu na zbiorze testowym:')
print(f'   Accuracy:  {accuracy_score(y_test, y_pred_best):.4f}')
print(f'   Precision: {precision_score(y_test, y_pred_best, zero_division=0):.4f}')
print(f'   Recall:    {recall_score(y_test, y_pred_best, zero_division=0):.4f}')
print(f'   F1:        {f1_score(y_test, y_pred_best, zero_division=0):.4f}')

print('\n🎉 Gratulacje! Ukończyłeś/aś bonus challenge!')

In [ ]:
# @title 💡 Podpowiedź — Bonus
print("""
💡 Podpowiedzi:

1. Siatka hiperparametrów:
   param_grid = {
       'n_estimators': [50, 100, 200],
       'max_depth': [5, 10, 20, None],
       'min_samples_split': [2, 5, 10]
   }

2. GridSearchCV:
   grid_search = GridSearchCV(
       estimator=RandomForestClassifier(random_state=42),
       param_grid=param_grid,
       cv=5,
       scoring='f1',
       n_jobs=-1,
       verbose=1
   )

3. Najlepszy wynik CV:
   best_cv_score = grid_search.best_score_
""")

In [ ]:
# @title ✅ Test — Bonus
try:
    assert best_cv_score is not ..., '❌ Nie obliczono best_cv_score'
    assert isinstance(best_cv_score, float), '❌ best_cv_score powinno być typu float'
    assert 0 < best_cv_score <= 1.0, f'❌ best_cv_score poza zakresem: {best_cv_score}'
    assert hasattr(grid_search, 'best_params_'), '❌ grid_search nie ma best_params_ — czy fit() się wykonał?'

    print(f'✅ Bonus zaliczony! Najlepszy CV score: {best_cv_score:.4f} 🏆🎉')
    print(f'📊 Najlepsze parametry: {grid_search.best_params_}')
    print(f'\n🎊 Brawo! Ukończyłeś/aś kompletny pipeline ML — od brudnych danych do modelu z tuningiem!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! Dasz radę 💪')
except Exception as e:
    print(f'❌ Coś poszło nie tak: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## 📝 Podsumowanie

Gratulacje — ukończyłeś/aś Laboratorium 4! 🎉🎓

### Co przećwiczyliśmy:

| Krok | Temat | Narzędzia |
|------|-------|-----------|
| 1 | Eksploracja danych | `shape`, `dtypes`, `isnull()`, `describe()` |
| 2 | Obsługa braków | `SimpleImputer` (median, most_frequent) |
| 3 | Czyszczenie danych | `replace()`, maski boolean, obsługa outliers |
| 4 | Feature Engineering | `pd.cut()`, `pd.get_dummies()`, tworzenie nowych cech |
| 5 | Przygotowanie do modelu | `train_test_split`, `StandardScaler`, `ColumnTransformer` |
| 6 | Budowanie modeli | `LogisticRegression`, `DecisionTree`, `RandomForest` |
| 7 | Ewaluacja | Confusion matrix, ROC curve, feature importance |
| 🏆 | Bonus — tuning | `GridSearchCV`, cross-validation |

### 🔑 Kluczowe wnioski:

1. **Przygotowanie danych to fundament** — nawet najlepszy algorytm nie pomoże, jeśli dane są brudne
2. **Pipeline** pozwala automatyzować cały proces przetwarzania
3. **Porównujcie modele** — nie zakładajcie z góry, który będzie najlepszy
4. **Accuracy to nie wszystko** — w problemach medycznych recall (czułość) jest kluczowy
5. **Cross-validation** daje bardziej wiarygodną ocenę niż pojedynczy split

### 📚 Materiały dodatkowe:

- [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html)
- [Kaggle — Data Cleaning Course](https://www.kaggle.com/learn/data-cleaning)
- [Google ML Crash Course — Dane i przygotowanie](https://developers.google.com/machine-learning/crash-course)

---

💡 **Na następnych zajęciach:** Zaawansowane techniki ML — ensemble methods, regularyzacja i optymalizacja Pipeline!

Do zobaczenia! 👋